# Notebook 12 — The Wave Equation: A Drum Membrane

Notebook 08 used the Laplacian to simulate diffusion: $\partial u/\partial t
= \alpha\nabla^2 u$, first order in time, and every point drifts toward its
neighborhood average until the whole field goes flat. Swap in a *second*
time derivative instead of a first,

$$\frac{\partial^2 u}{\partial t^2} = c^2 \nabla^2 u$$

and the qualitative behavior flips completely: instead of smoothing out,
disturbances **propagate and reflect** — this is the equation for a
vibrating drum membrane, a plucked string generalized to 2D, or (mathematically
identical) a pressure wave. Same operator, same `laplacian_2d`, entirely
different physics, because a second-order-in-time equation stores momentum
and a first-order one doesn't.

In [1]:
# Make src/ importable from inside notebooks/
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
%matplotlib inline

from src import operators as ops


## 1. By hand

**Discretizing in time.** The same central-difference idea from notebook
06, applied to a *second* derivative this time:

$$\frac{\partial^2 u}{\partial t^2}\bigg|_{t_n} \approx \frac{u^{n+1} - 2u^n + u^{n-1}}{\Delta t^2}$$

Setting this equal to $c^2\nabla^2 u^n$ and solving for the only unknown,
$u^{n+1}$:

$$u^{n+1} = 2u^n - u^{n-1} + c^2\Delta t^2\,\nabla^2 u^n$$

This is **leapfrog** time-stepping: unlike notebook 08's forward Euler,
advancing one step needs the *two* previous time levels, not one — a direct
consequence of the equation itself being second order in time. That means
the initial condition also needs two pieces of information: a starting
shape $u^0$ *and* an initial velocity $v^0=\partial u/\partial t|_0$ (a
plucked drum released from rest has $v^0=0$; a struck one doesn't). The
first step is special since there's no $u^{-1}$ to leapfrog from — a
Taylor expansion in time gives

$$u^1 \approx u^0 + \Delta t\, v^0 + \frac{\Delta t^2}{2}c^2\nabla^2 u^0$$

**Boundary condition.** A drum's rim is clamped — physically pinned at
zero displacement, not free to move. That's a **Dirichlet** condition,
$u=0$ on the boundary for all $t$, imposed by simply never updating the
boundary values (they're initialized to $0$ and the leapfrog update only
ever touches interior points). This is a different choice than notebook
08's heat simulation, which left the boundary to whatever `np.gradient`'s
one-sided differences implied, or notebook 10's periodic wraparound —
three different notebooks, three different legitimate boundary conditions
for three different physical setups.

**Stability.** Leapfrog with a 5-point Laplacian is only conditionally
stable; the 2D CFL condition is

$$c\,\Delta t \le \frac{\Delta x}{\sqrt{2}}$$

(the direct 2D generalization of the classic 1D wave-equation CFL limit
$c\Delta t \le \Delta x$) — same Taylor-expansion accounting as notebook
08's stability bound, just for a different scheme.

## 2. A known solution to check against

A rectangular membrane $[0,L]\times[0,L]$, clamped on all sides, has exact
standing-wave solutions (the drum's normal modes):

$$u(x,y,t) = \sin\!\left(\frac{m\pi x}{L}\right)\sin\!\left(\frac{n\pi y}{L}\right)\cos(\omega_{mn} t), \qquad \omega_{mn} = c\pi\sqrt{\frac{m^2+n^2}{L^2}}$$

Each already satisfies $u=0$ on the boundary by construction (since $\sin$
vanishes at $x=0,L$ and $y=0,L$). The fundamental mode $m=n=1$ gives a
single frequency to check the simulation against — release the membrane
from rest in exactly this shape, and it should oscillate at exactly
$\omega_{11}$, forever, with no distortion.

In [2]:
L = 1.0
c = 1.0
n = 101

x_vals = np.linspace(0, L, n)
y_vals = np.linspace(0, L, n)
X, Y = np.meshgrid(x_vals, y_vals)
dx = x_vals[1] - x_vals[0]
dy = y_vals[1] - y_vals[0]

dt_max = dx / (c * np.sqrt(2))
dt = 0.9 * dt_max

omega_11 = c * np.pi * np.sqrt(2) / L
period_11 = 2 * np.pi / omega_11
n_periods = 5
n_steps = int(n_periods * period_11 / dt)

print(f"dt = {dt:.5f}  (stability limit: {dt_max:.5f})")
print(f"analytical omega_11 = {omega_11:.4f}, period = {period_11:.4f}")
print(f"simulating {n_periods} periods over {n_steps} steps")


dt = 0.00636  (stability limit: 0.00707)
analytical omega_11 = 4.4429, period = 1.4142
simulating 5 periods over 1111 steps


In [3]:
u0 = np.sin(np.pi * X / L) * np.sin(np.pi * Y / L)  # fundamental mode, v0 = 0

u_prev = u0.copy()
lap0 = ops.laplacian_2d(u_prev, dx, dy)
u = u_prev.copy()
u[1:-1, 1:-1] = u_prev[1:-1, 1:-1] + 0.5 * (c * dt) ** 2 * lap0[1:-1, 1:-1]

center = (n // 2, n // 2)
trace = [u_prev[center], u[center]]

for step in range(2, n_steps + 1):
    lap = ops.laplacian_2d(u, dx, dy)
    u_next = np.zeros_like(u)
    u_next[1:-1, 1:-1] = (
        2 * u[1:-1, 1:-1] - u_prev[1:-1, 1:-1] + (c * dt) ** 2 * lap[1:-1, 1:-1]
    )
    u_prev, u = u, u_next
    trace.append(u[center])

trace = np.array(trace)
print(f"max|u| over the run: {np.max(np.abs(trace)):.4f}  (started at 1.0 -- no blow-up, no visible decay)")


max|u| over the run: 1.0000  (started at 1.0 -- no blow-up, no visible decay)


In [4]:
# Recover the oscillation frequency from the simulated trace via FFT,
# and compare to the analytical omega_11 -- the same "measure, don't
# assume" check as notebook 06's convergence-order measurements.
spectrum = np.abs(np.fft.rfft(trace - trace.mean()))
freqs = np.fft.rfftfreq(len(trace), d=dt)
peak_idx = np.argmax(spectrum[1:]) + 1
omega_measured = 2 * np.pi * freqs[peak_idx]

rel_err = abs(omega_measured - omega_11) / omega_11
print(f"measured omega = {omega_measured:.4f}  (analytical {omega_11:.4f}, relative error {rel_err:.2%})")


measured omega = 4.4393  (analytical 4.4429, relative error 0.08%)


## 3. Does leapfrog conserve energy?

A drum losing no energy to friction should oscillate forever at constant
amplitude. The membrane's total mechanical energy is kinetic plus elastic
potential energy:

$$E = \frac{1}{2}\int\!\!\int \left[\left(\frac{\partial u}{\partial t}\right)^2 + c^2|\nabla u|^2\right] dA$$

Notebook 08's heat simulation lost about 13% of its total heat to the
(imperfectly insulated) boundary over its run — worth checking whether
this scheme does anything similarly leaky, rather than assuming leapfrog
is well-behaved just because it's a standard method.

In [5]:
def membrane_energy(u_now, u_before, dt, dx, dy, c):
    v = (u_now - u_before) / dt
    du_dx, du_dy = ops.gradient_2d(u_now, dx, dy)
    kinetic = 0.5 * np.sum(v**2) * dx * dy
    potential = 0.5 * c**2 * np.sum(du_dx**2 + du_dy**2) * dx * dy
    return kinetic + potential


u_prev = u0.copy()
lap0 = ops.laplacian_2d(u_prev, dx, dy)
u = u_prev.copy()
u[1:-1, 1:-1] = u_prev[1:-1, 1:-1] + 0.5 * (c * dt) ** 2 * lap0[1:-1, 1:-1]

energies = []
for step in range(2, n_steps + 1):
    lap = ops.laplacian_2d(u, dx, dy)
    u_next = np.zeros_like(u)
    u_next[1:-1, 1:-1] = (
        2 * u[1:-1, 1:-1] - u_prev[1:-1, 1:-1] + (c * dt) ** 2 * lap[1:-1, 1:-1]
    )
    if step % 50 == 0:
        energies.append(membrane_energy(u_next, u, dt, dx, dy, c))
    u_prev, u = u, u_next

energies = np.array(energies)
drift_pct = 100 * (energies[-1] - energies[0]) / energies[0]
spread_pct = 100 * energies.std() / energies.mean()
print(f"energy: start={energies[0]:.4f}  end={energies[-1]:.4f}  (drift {drift_pct:+.1f}%)")
print(f"energy oscillates with std/mean = {spread_pct:.1f}% around its average, rather than draining one-directionally")


energy: start=2.4565  end=2.5322  (drift +3.1%)
energy oscillates with std/mean = 1.3% around its average, rather than draining one-directionally


Unlike notebook 08's monotonic, one-directional heat loss, this
energy **oscillates** within a few percent of its mean rather than
draining away — leapfrog is a symplectic-adjacent scheme that doesn't
dissipate energy the way forward Euler does. It isn't *exactly* conserved
(the velocity estimate here is itself a finite difference, `O(Δt)`
accurate, and the potential-energy gradient near the clamped boundary
picks up some of notebook 06's one-sided-difference error too) — but
"oscillates around a stable mean" and "drains away" are qualitatively
different failure modes, and only actually measuring it tells you which
one you have.

## 4. Visualization: a plucked membrane

The eigenmode above is the cleanest thing to *verify* against, but a
single smooth mode is a boring thing to *watch*. Pluck the membrane
off-center instead — a localized bump released from rest — and watch the
resulting ripple spread out and bounce off the clamped edges.

In [6]:
n_anim = 80
x_a = np.linspace(0, L, n_anim)
y_a = np.linspace(0, L, n_anim)
Xa, Ya = np.meshgrid(x_a, y_a)
dxa, dya = x_a[1] - x_a[0], y_a[1] - y_a[0]
dt_a = 0.9 * dxa / (c * np.sqrt(2))

pluck = 0.6 * np.exp(-((Xa - 0.3) ** 2 + (Ya - 0.3) ** 2) / (2 * 0.06 ** 2))
pluck[0, :] = pluck[-1, :] = pluck[:, 0] = pluck[:, -1] = 0.0  # start clamped, consistent with the BC

u_prev_a = pluck.copy()
lap0_a = ops.laplacian_2d(u_prev_a, dxa, dya)
u_a = u_prev_a.copy()
u_a[1:-1, 1:-1] = u_prev_a[1:-1, 1:-1] + 0.5 * (c * dt_a) ** 2 * lap0_a[1:-1, 1:-1]

total_time = 3.0
n_steps_a = int(total_time / dt_a)
save_every = 10

frames = [u_prev_a.copy(), u_a.copy()]
for step in range(2, n_steps_a + 1):
    lap_a = ops.laplacian_2d(u_a, dxa, dya)
    u_next_a = np.zeros_like(u_a)
    u_next_a[1:-1, 1:-1] = (
        2 * u_a[1:-1, 1:-1] - u_prev_a[1:-1, 1:-1] + (c * dt_a) ** 2 * lap_a[1:-1, 1:-1]
    )
    u_prev_a, u_a = u_a, u_next_a
    if step % save_every == 0:
        frames.append(u_a.copy())

print(f"{len(frames)} frames over {n_steps_a} steps ({total_time} time units)")


39 frames over 372 steps (3.0 time units)


In [7]:
fig, ax = plt.subplots(figsize=(6, 5.5))
vmax = np.max(np.abs(frames[0]))
im = ax.imshow(
    frames[0], extent=(0, L, 0, L), origin="lower",
    cmap="RdBu_r", vmin=-vmax, vmax=vmax,
)
title = ax.set_title("t = 0.00")
fig.colorbar(im, ax=ax, label="u (membrane displacement)")


def update(i):
    im.set_data(frames[i])
    title.set_text(f"t = {i * save_every * dt_a:.2f}")
    return im, title


anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=60, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


**Reading this.** The pluck spreads outward as an expanding ring,
reflects off the clamped edges (flipping sign on reflection, the way a
wave does at a fixed end — watch for red turning to blue after the first
bounce), and keeps interfering with its own reflections rather than ever
settling down, since nothing in this simulation removes energy. Compare
to notebook 08's animation: that one converges toward a single flat blob;
this one just keeps ringing.

## 5. Exercise

Change the initial condition to a **struck** membrane instead of a
plucked one: keep `pluck` at zero everywhere ($u^0=0$), but give it a
nonzero initial velocity $v^0$ instead — e.g. a localized bump of velocity
at the center, using the Taylor-expansion first step
$u^1 \approx u^0 + \Delta t\,v^0 + \tfrac{\Delta t^2}{2}c^2\nabla^2 u^0$
with $u^0=0$. This is physically a mallet strike rather than a finger
pluck. Does the resulting ripple pattern look different from this
notebook's example, or just scaled?

Then try changing just `c` (the wave speed) and predict, before running
it, whether the CFL-limited `dt` should get larger or smaller — and
whether the pattern in the animation moves faster or slower as a result.


In [8]:
# Your answer here.
# u0_struck = np.zeros_like(Xa)
# ...
